# Bama inventory — EDA overviewThis notebook is a **thin viewer** over `bama_eda`. Every number it shows comes froma tested package function; nothing is computed only here. That is deliberate — ananalysis whose logic lives in notebook cells cannot be tested, reviewed or rerun.Read `README_EDA.md` first, and in particular the terminology section: this datasetmeasures **time to disappearance**, never time to sale.

In [ ]:
import syssys.path.insert(0, "../src")import pandas as pdfrom bama_eda.config import DEFAULT_DEEP_SQLITE, EdaConfigfrom bama_eda.database import ReadOnlyDatabasefrom bama_eda.provenance import Manifest, build_context, select_runfrom bama_eda.dataset_builder import build_cross_section, build_panel, build_vehicle_datasetpd.set_option("display.max_columns", 60)pd.set_option("display.width", 200)

## 1. Select the analysis runThe latest run that is valid, finished, applied its comparison and persisted a non-empty inventory.

In [ ]:
cfg = EdaConfig(database_url="sqlite:///../monitor_data/bama_monitor.sqlite")cfg.enrichment.enabled = Truecfg.enrichment.database_path = DEFAULT_DEEP_SQLITEdb = ReadOnlyDatabase(cfg.database_url)run, warnings = select_run(db, "latest-valid")context = build_context(db, run)manifest = Manifest(context, cfg)context.as_dict()

## 2. Build the three datasetsDifferent grains, never conflated: advertisement, (run × advertisement), physical vehicle.

In [ ]:
cross = build_cross_section(db, context, cfg, manifest)panel = build_panel(db, context, cfg, manifest)vehicles = build_vehicle_dataset(db, cross, context, cfg, manifest)pd.DataFrame([    {"dataset": "A advertisement_cross_section", "rows": len(cross), "columns": cross.shape[1]},    {"dataset": "B longitudinal_panel", "rows": len(panel), "columns": panel.shape[1]},    {"dataset": "C vehicle_entity_dataset", "rows": len(vehicles), "columns": vehicles.shape[1]},])

## 3. Integrity auditIf anything critical fires, stop: the rest of the notebook would be describing a dataset that is wrong.

In [ ]:
from bama_eda.validation import audit, summarisedatasets = {    "advertisement_cross_section": cross,    "longitudinal_panel": panel,    "vehicle_entity_dataset": vehicles,}findings = audit(db, datasets, context, cfg)summary = summarise(findings)print("passed:", summary["passed"], "|", summary["checks_run"], "checks,",      summary["checks_with_findings"], "with findings")pd.DataFrame([f.as_dict() for f in findings if f.count]) or "no findings"

## 4. What is in the inventory

In [ ]:
from bama_eda.cross_sectional import composition, inventory_summaryinventory = inventory_summary(cross, panel, vehicles)pd.Series(inventory)

In [ ]:
comp = composition(cross, min_group_size=cfg.thresholds.min_group_size)comp[comp.dimension == "brand"].head(12)

## 5. PriceUnit is **toman**, never converted. These are **asking** prices, not transaction prices.

In [ ]:
from bama_eda.profiling import profile_frameprofile_frame(cross)[["variable","count","missing","median","p25","p75","max","skewness","note"]].head(6)

In [ ]:
from bama_eda.price_analysis import price_by_segmentseg = price_by_segment(cross, min_group_size=cfg.thresholds.min_group_size)seg[seg.dimension == "brand"][    ["segment_value","listing_count","median_price","p25_price","p75_price","sufficient_sample"]].head(10)

### What moves with priceSpearman is the coefficient to read: prices are strongly skewed and the relationships are monotone rather than linear. Compare the two columns — where they disagree, Pearson is the misleading one.

In [ ]:
from bama_eda.correlations import correlation_matrix, price_association_summarycorr = correlation_matrix(cross, min_pairs=cfg.thresholds.min_correlation_pairs)summary = price_association_summary(corr)pd.DataFrame(summary["ranked"])

## 6. Disappearance and censoringThe event is **disappearance from the observed inventory**. Not a sale.

In [ ]:
from bama_eda.duration_analysis import build_duration_table, duration_summary, kaplan_meierdurations = build_duration_table(cross)pd.Series(duration_summary(durations)["censoring_classes"])

In [ ]:
km = kaplan_meier(    durations,    min_subjects=cfg.thresholds.min_survival_group_size,    min_events=cfg.thresholds.min_survival_events,)print("curve available:", km["available"])print(km.get("note", ""))

## 7. What this dataset cannot answerThe most useful output of the analysis. Check these before quoting any figure.

In [ ]:
from bama_eda.temporal_analysis import duration_ranking_feasibility, publication_coveragepub = publication_coverage(cross)print(pub["interpretation"])print()print(duration_ranking_feasibility(cross, min_eligible=cfg.thresholds.min_survival_group_size)["note"])

In [ ]:
from bama_eda.report_builder import _unsupported_conclusionsfor item in _unsupported_conclusions({}):    print("-", item)

## 8. Full reportEverything above plus charts, tables and the manifest:```bashpython -m bama_eda run \  --database-url "sqlite:///monitor_data/bama_monitor.sqlite" \  --run-id latest-valid --enrich-attributes```

In [ ]:
db.close()